<a href="https://colab.research.google.com/github/dee0742/ML-FlyRank-Task/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dee0742/ML-FlyRank-Task/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My baseline rule

I will rank pages higher when they have been unchanged for longer and have higher search demand.

The rule uses two signals:

- `days_since_last_update` — older pages receive a higher staleness score.
- `search_volume` — pages with higher search demand receive a higher demand score.

The final score is the average of the two normalized scores.

This is a simple decision-support rule. It is not a prediction that a page will decline.

### Reason codes

- `STALE_HIGH_DEMAND` — page is old and has meaningful search demand.
- `STALE_LOW_DEMAND` — page is old but has lower search demand.
- `FRESH_HIGH_DEMAND` — page is relatively fresh but has meaningful search demand.
- `FRESH_LOW_DEMAND` — page is relatively fresh and has lower search demand.

In [7]:
import numpy as np
import pandas as pd
import os

work = df.copy()

# ---------------------------------------------------------
# SIGNAL 1: STALENESS
# ---------------------------------------------------------

work["staleness_bucket"] = pd.cut(
    work["days_since_last_update"],
    bins=[-1, 30, 90, 180, 365, np.inf],
    labels=["0-30", "31-90", "91-180", "181-365", "365+"]
)

staleness_check = (
    work.groupby("staleness_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        decline_rate=("trend_direction",
                      lambda x: (x == "down").mean())
    )
    .reset_index()
)

print("SIGNAL 1 — STALENESS")
print(staleness_check.to_string(index=False))


# ---------------------------------------------------------
# SIGNAL 2: SEARCH VOLUME
# ---------------------------------------------------------

work["volume_bucket"] = pd.cut(
    work["search_volume"],
    bins=[-1, 100, 500, 1000, 5000, np.inf],
    labels=["0-100", "101-500", "501-1000", "1001-5000", "5000+"]
)

volume_check = (
    work.groupby("volume_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        decline_rate=("trend_direction",
                      lambda x: (x == "down").mean())
    )
    .reset_index()
)

print("\nSIGNAL 2 — SEARCH VOLUME")
print(volume_check.to_string(index=False))

SIGNAL 1 — STALENESS
staleness_bucket     n  decline_rate
            0-30 20480      0.511377
           31-90   175      0.588571
          91-180  9171      0.611057
         181-365   169      0.467456
            365+     5      0.600000

SIGNAL 2 — SEARCH VOLUME
volume_bucket     n  decline_rate
        0-100 24483      0.573010
      101-500  2021      0.513607
     501-1000   468      0.446581
    1001-5000   408      0.421569
        5000+   152      0.506579


### Signal verdicts

**Staleness — [CONFIRMED / OPPOSITE / MIXED / FALSE]**

The bucket table shows that the observed decline rate [increases/decreases/does not move consistently] as days since the last update increases. Therefore, the relationship is [directional evidence/supports the rule/etc.].

**Search volume — [CONFIRMED / OPPOSITE / MIXED / FALSE]**

The bucket table shows that the observed decline rate [increases/decreases/does not move consistently] across search-volume buckets. Therefore, the relationship is [directional evidence/supports the rule/etc.].

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Scoring rule

The baseline score is the average of:

1. percentile rank of `days_since_last_update`
2. percentile rank of `search_volume`

A higher score means the page has a combination of higher staleness and higher search demand.

The rule does not use `trend_direction` or `trend_pct`.

In [8]:
# ---------------------------------------------------------
# BUILD BASELINE SCORE
# ---------------------------------------------------------

baseline = df.copy()

# Staleness score: higher = more stale
baseline["staleness_score"] = (
    baseline["days_since_last_update"].rank(pct=True)
)

# Demand score: higher = more search demand
baseline["demand_score"] = (
    baseline["search_volume"]
    .rank(pct=True, na_option="bottom")
)

# Equal-weight transparent score
baseline["action_score"] = (
    0.5 * baseline["staleness_score"]
    + 0.5 * baseline["demand_score"]
)


# ---------------------------------------------------------
# REASON CODE
# ---------------------------------------------------------

stale = baseline["days_since_last_update"] >= 180
high_demand = baseline["search_volume"].fillna(0) >= 500

baseline["reason_code"] = np.select(
    [
        stale & high_demand,
        stale & ~high_demand,
        ~stale & high_demand
    ],
    [
        "STALE_HIGH_DEMAND",
        "STALE_LOW_DEMAND",
        "FRESH_HIGH_DEMAND"
    ],
    default="FRESH_LOW_DEMAND"
)


# ---------------------------------------------------------
# ACTION LABEL
# ---------------------------------------------------------

baseline["action"] = np.where(
    stale & high_demand,
    "REVIEW_REFRESH",
    "MONITOR"
)


# ---------------------------------------------------------
# RANK
# ---------------------------------------------------------

baseline = (
    baseline
    .sort_values("action_score", ascending=False)
    .reset_index(drop=True)
)

baseline["rank"] = baseline.index + 1


# ---------------------------------------------------------
# WRITE CSV
# ---------------------------------------------------------

output_cols = [
    "rank",
    "content_id",
    "action_score",
    "reason_code",
    "action",
    "days_since_last_update",
    "search_volume"
]

queue = baseline[output_cols]

os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Rows ranked:", len(queue))
print("\nTop 20:")
print(queue.head(20).to_string(index=False))

print("\nCSV written to:")
print("work/outputs/baseline_action_score.csv")

Rows ranked: 30000

Top 20:
 rank           content_id  action_score      reason_code  action  days_since_last_update  search_volume
    1 content_f01216059a6a      0.979358 STALE_LOW_DEMAND MONITOR                     335            NaN
    2 content_e2b702f4f92b      0.979333 STALE_LOW_DEMAND MONITOR                     334            NaN
    3 content_06e19c6486b0      0.979333 STALE_LOW_DEMAND MONITOR                     334            NaN
    4 content_f488400fca67      0.979167 STALE_LOW_DEMAND MONITOR                     305            NaN
    5 content_df1fa766cac2      0.979033 STALE_LOW_DEMAND MONITOR                     304            NaN
    6 content_07ce98c6085a      0.979033 STALE_LOW_DEMAND MONITOR                     304            NaN
    7 content_15fe075b97bc      0.979033 STALE_LOW_DEMAND MONITOR                     304            NaN
    8 content_ab27c30d81f4      0.979033 STALE_LOW_DEMAND MONITOR                     304            NaN
    9 content_84d12054c0c0 

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*


The review below treats the baseline as decision support rather than as a prediction.

For each page, I record:

- the recommended action,
- the reason code,
- why the rule ranked it highly,
- and what could make the recommendation wrong.

In [9]:
top20 = baseline.head(20).copy()

def confidence_note(row):
    if row["reason_code"] == "STALE_HIGH_DEMAND":
        return "Both staleness and demand support the review."
    elif row["reason_code"] == "STALE_LOW_DEMAND":
        return "Staleness supports review, but demand is weaker."
    elif row["reason_code"] == "FRESH_HIGH_DEMAND":
        return "Demand supports attention, but the page is relatively fresh."
    else:
        return "Both signals provide weaker evidence for immediate refresh."

def what_makes_wrong(row):
    if row["reason_code"] == "STALE_HIGH_DEMAND":
        return "The content may still be current despite being old."
    elif row["reason_code"] == "STALE_LOW_DEMAND":
        return "Low demand may mean an update has limited practical value."
    elif row["reason_code"] == "FRESH_HIGH_DEMAND":
        return "The page was recently updated, so another refresh may be unnecessary."
    else:
        return "The page has weak evidence for immediate refresh."

top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    what_makes_wrong,
    axis=1
)

review = top20[
    [
        "rank",
        "content_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

print(review.to_string(index=False))

 rank           content_id  action      reason_code                                  confidence_note                                   what_would_make_it_wrong
    1 content_f01216059a6a MONITOR STALE_LOW_DEMAND Staleness supports review, but demand is weaker. Low demand may mean an update has limited practical value.
    2 content_e2b702f4f92b MONITOR STALE_LOW_DEMAND Staleness supports review, but demand is weaker. Low demand may mean an update has limited practical value.
    3 content_06e19c6486b0 MONITOR STALE_LOW_DEMAND Staleness supports review, but demand is weaker. Low demand may mean an update has limited practical value.
    4 content_f488400fca67 MONITOR STALE_LOW_DEMAND Staleness supports review, but demand is weaker. Low demand may mean an update has limited practical value.
    5 content_df1fa766cac2 MONITOR STALE_LOW_DEMAND Staleness supports review, but demand is weaker. Low demand may mean an update has limited practical value.
    6 content_07ce98c6085a MONITOR STALE

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

A potentially weak pick is a page that receives a high ranking even though one of the two signals provides weak support.

For example, a page can rank highly because it is very stale even when its search demand is low. This is a limitation of using a simple equal-weight score.

### Leakage check

The baseline score uses only:

- `days_since_last_update`
- `search_volume`

It does not use `trend_direction` or `trend_pct`.

Those fields are outcome-related and therefore are excluded from the baseline features.

In [10]:
# ---------------------------------------------------------
# WEAK PICKS
# ---------------------------------------------------------

weak_picks = baseline[
    (baseline["action_score"] >= baseline["action_score"].quantile(0.95)) &
    (
        (baseline["search_volume"].fillna(0) < 500) |
        (baseline["days_since_last_update"] < 180)
    )
].head(5)

print("Potential weak picks:")
print(
    weak_picks[
        [
            "rank",
            "content_id",
            "action_score",
            "reason_code",
            "action",
            "days_since_last_update",
            "search_volume"
        ]
    ].to_string(index=False)
)


# ---------------------------------------------------------
# LEAKAGE CHECK
# ---------------------------------------------------------

used_features = [
    "days_since_last_update",
    "search_volume"
]

forbidden_features = [
    "trend_direction",
    "trend_pct"
]

print("\nBaseline features:")
print(used_features)

print("\nLeakage check:")

for feature in forbidden_features:
    print(
        f"{feature}:",
        "NOT USED" if feature not in used_features else "LEAKAGE"
    )

Potential weak picks:
 rank           content_id  action_score      reason_code  action  days_since_last_update  search_volume
    1 content_f01216059a6a      0.979358 STALE_LOW_DEMAND MONITOR                     335            NaN
    2 content_e2b702f4f92b      0.979333 STALE_LOW_DEMAND MONITOR                     334            NaN
    3 content_06e19c6486b0      0.979333 STALE_LOW_DEMAND MONITOR                     334            NaN
    4 content_f488400fca67      0.979167 STALE_LOW_DEMAND MONITOR                     305            NaN
    5 content_df1fa766cac2      0.979033 STALE_LOW_DEMAND MONITOR                     304            NaN

Baseline features:
['days_since_last_update', 'search_volume']

Leakage check:
trend_direction: NOT USED
trend_pct: NOT USED


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.